# NiVe1303 → OSNet: контролируемый перенос

Первый этап использования внешних данных, без масок частей кузова и без изменения архитектуры OSNet.

- **B0**: stock → наши данные, 1700 шагов.
- **C1**: stock → наши данные, 850 шагов → сброс classifier/optimizer/LR → наши данные, 1700 шагов.
- **E1**: stock → NiVe train, 850 шагов → такой же сброс → наши данные, 1700 шагов.

C1 и E1 сопоставимы по числу шагов и размеру batch, не по wall time. Backbone и BNNeck (включая running statistics) переносятся. Классификатор identity и optimizer не переносятся. Предобучение заканчивается на фиксированном последнем шаге, без выбора по test.

По умолчанию — **pilot**, один seed, 5 стадий обучения, 6800 updates суммарно. Никакой outer validation, экспорта или замены MVP. Полное обучение запускаешь ты. [README](README.md).


In [ ]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/nive_transfer.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из проекта Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import torch
import pandas as pd
from IPython.display import Markdown, display
from training.nive_transfer import SOURCE_URL, ARMS, prepare, run_experiment
from training.osnet_ablation_suite import Budget

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
print('Repository:', REPO)
print('Torch:', torch.__version__)


## 1. Источник и настройки

Подтверди, что локальная копия скачана с указанной официальной страницы **NiVe1303 v1**, установив `SOURCE_CONFIRMED = True`. Это подтверждение пользователя, а не результат догадки по названию папки. Если источник другой — оставь False и сначала укажи его для проверки воспроизводимости.

Условия доступа: CC BY 4.0, автор Ruozheng LI, DOI `10.17632/42wv2svztx.1`. Исходный архив отсутствует, поэтому manifest фиксирует SHA256 каждого JPEG. Номера автор описывает как неразличимые из-за смазывания, не как обязательно закрытые маской. OCR и номерные признаки не используются.

Сначала дождись окончания variant 14; новый запуск дополнительно проверяет его lock. Не запускай две GPU-серии одновременно.


In [ ]:
RUN_NAME = 'nive_pilot_v1'
PHASE = 'pilot'  # После анализа пилота: 'confirm' с ТЕМ ЖЕ RUN_NAME и прочими настройками.
SOURCE_CONFIRMED = False  # True только после подтверждения источника локальной копии.
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
ALLOW_CPU_TRAINING = False
PRETRAIN_BUDGET = Budget(max_steps=850, evaluation_interval=200, warmup_steps=100)
TARGET_BUDGET = Budget(max_steps=1700, evaluation_interval=200, warmup_steps=100)
SEEDS = (20260915, 20260916, 20260917)
NIVE_ROOT = REPO / 'NiVe1303'

print('Confirm source:', SOURCE_URL)
print('Phase:', PHASE, '| device:', DEVICE)
display(pd.DataFrame([{'arm': name, 'training': description} for name, description in ARMS.items()]))
print('Pilot upper bound:', 3 * TARGET_BUDGET.max_steps + 2 * PRETRAIN_BUDGET.max_steps, 'updates')
print('Full confirmation including pilot upper bound:',
      21 * TARGET_BUDGET.max_steps + 10 * PRETRAIN_BUDGET.max_steps, 'updates')


## 2. Preflight — без обучения

Проверяется исходный train организаторов (CSV, 9556 изображений, splits, stock/MVP и evaluator), а также 31 735 фотографий NiVe. NiVe train/test должны быть разделены по identity; побайтовые дубликаты и совпадения с train организаторов блокируют запуск. Это не полноценная проверка near-duplicates.

Split задаётся только папками фотографий. `train_MK_PURE` содержит также маски 5049 внешних test-фотографий, поэтому маски в этой серии вообще не читаются. Все 600 внешних test identity остаются holdout. Префиксы EN/ES/N/S — только proxy ракурса для PK-sampling, не подтверждённые физические camera ID. Имена файлов, время и камера не передаются модели.

Manifest и результаты записываются только в новый `variant_15_nive_transfer/runs/<RUN_NAME>`. Исходные фото, bbox, CSV и старые серии не изменяются.


In [ ]:
if not SOURCE_CONFIRMED:
    raise RuntimeError('Подтверди источник локальной NiVe1303 v1: проверь SOURCE_URL и поставь SOURCE_CONFIRMED=True. Если источник другой — сначала уточни его.')
context = prepare(RUN_NAME, device=DEVICE, pretrain_budget=PRETRAIN_BUDGET,
                  target_budget=TARGET_BUDGET, seeds=SEEDS, source_confirmed=SOURCE_CONFIRMED,
                  nive=NIVE_ROOT)
print('Output:', context['output'])
print('Fingerprint:', context['signature'])
print('Organizer rows:', len(context['rows']))
display(pd.DataFrame(context['manifest']['nive']['counts']).T)
print('Training has not started yet.')


## 3. Запуск

`pilot`: все три варианта на первом seed, 5 стадий, до 6800 updates. B0 в этой отдельной серии обучается заново, а не переносится через неподтверждённую совместимость старых checkpoint. Пилот даёт сигнал для обсуждения, не доказательство улучшения.

`confirm`: продолжает тот же run; завершённые стадии не переобучаются. Все три варианта проходят три primary seed; выбор и число target refit steps фиксируются до alternate. Затем все три проходят три alternate seed и final refit на исходном outer train. NiVe-предобучение одного seed переиспользуется между folds, поскольку не видит данные организаторов; C1 предобучается отдельно внутри каждого fold.

Полный confirm: до 31 уникальной стадии / 44 200 updates **включая pilot**, не ещё 31 поверх него. Порог выбирается только на исходной calibration; original/masked validation и ONNX-export выполняются после freeze. Изменение PHASE не меняет manifest; изменение бюджета, seed, данных или кода требует нового RUN_NAME.

После прерывания: Restart Kernel → Run All с прежними настройками. Повторится не более незавершённого блока; прогресс показывает stage/arm/seed, шаги, loss через history.json, mAP, elapsed и ETA. Не меняй код во время обучения.


In [ ]:
if DEVICE == 'cpu' and not ALLOW_CPU_TRAINING:
    raise RuntimeError('CPU-обучение отключено. Выбери MPS/CUDA или осознанно ALLOW_CPU_TRAINING=True.')
result = run_experiment(context, phase=PHASE)
print('Complete. Promoted:', result['promoted'], '| Outer evaluated:', result['outer_evaluated'])


## 4. Результат

Сравни E1 не только с B0, но и с C1: это разделяет эффект внешних данных и дополнительных optimizer updates. Для confirm доступны paired-seed deltas, alternate split, исходная validation, отказ и masked diagnostics с порогами original calibration. `paired_bootstrap.json` сравнивает E1 с обоими контролями; это условная оценка для фиксированной gallery, не независимый тест.

Рост на одном seed не гарантирует улучшения. Validation уже использовалась в исследованиях и остаётся development-набором. Ни одна модель не продвигается в MVP автоматически. Смешанное обучение, маски частей, отдельный night/day-срез и изменение архитектуры не входят в этот первый этап.


In [ ]:
report_name = 'PILOT_RESULTS.md' if PHASE == 'pilot' else 'RESULTS.md'
display(Markdown((context['output'] / report_name).read_text(encoding='utf-8')))
print('Detailed results:', context['output'])
